# Symplektisches Euler-Verfahren

Harmonischer Oszillator und Chirikov-Standardabbildung

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import Video, display

# Verzeichnis für exportierte Abbildungen und Videos
notebook_name = "SymplecticEuler"
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / ".git").exists()),
    Path.cwd(),
)
output_dir = project_root / "data" / "01_mechanik" / notebook_name
output_dir.mkdir(parents=True, exist_ok=True)

# Abbildungsstil für ein A4-Skript
latex_textwidth_in = 6.3
single_panel_width = 0.5 * latex_textwidth_in
single_panel_square = (single_panel_width, single_panel_width)
single_panel_wide = (single_panel_width, 0.68 * single_panel_width)
full_width_three_panel = (latex_textwidth_in, 0.36 * latex_textwidth_in)
animation_display_width = 360

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "lines.linewidth": 1.2,
    "axes.linewidth": 0.8,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print(f"Output directory: {output_dir}")

## Harmonischer Oszillator

Für

$$H(q,p)=\frac12\left(p^2+\omega^2q^2\right)$$

verwenden wir die Kick-Drift-Variante des symplektischen Euler-Verfahrens:

$$p_{n+1}=p_n-h\omega^2q_n, \qquad q_{n+1}=q_n+h p_{n+1}. $$

Das Verfahren erhält nicht exakt $H$, sondern die modifizierte Hamiltonfunktion

$$\widetilde H_h(q,p)=\frac12\left(p^2+\omega^2q^2-h\omega^2qp\right).$$

Für $h\omega<2$ sind ihre Niveaulinien Ellipsen. Wir wählen absichtlich einen großen Zeitschritt, damit die Neigung dieser Ellipsen deutlich sichtbar wird.

In [ ]:
def symplectic_euler_step(q, p, h, omega):
    p_new = p - h * omega**2 * q
    q_new = q + h * p_new
    return q_new, p_new


def solve_symplectic_euler(q0, p0, h, n_steps, omega=1.0):
    q = np.empty(n_steps + 1)
    p = np.empty(n_steps + 1)
    q[0], p[0] = q0, p0
    for n in range(n_steps):
        q[n + 1], p[n + 1] = symplectic_euler_step(q[n], p[n], h, omega)
    return q, p


def harmonic_energy(q, p, omega):
    return 0.5 * (p**2 + omega**2 * q**2)


def modified_harmonic_energy(q, p, h, omega):
    return 0.5 * (p**2 + omega**2 * q**2 - h * omega**2 * q * p)

In [ ]:
omega = 1.0
h = 0.5  # großer, aber noch stabiler Zeitschritt: h*omega < 2
n_steps = 240
q0, p0 = 1.0, 0.0

q, p = solve_symplectic_euler(q0, p0, h, n_steps, omega)
t = h * np.arange(n_steps + 1)
energy = harmonic_energy(q, p, omega)
modified_energy = modified_harmonic_energy(q, p, h, omega)
modified_energy_0 = modified_energy[0]
exact_energy_0 = energy[0]


In [ ]:
limit = 1.15 * max(np.max(np.abs(q)), np.max(np.abs(p)))
q_grid, p_grid = np.meshgrid(
    np.linspace(-limit, limit, 400),
    np.linspace(-limit, limit, 400),
)
modified_energy_grid = modified_harmonic_energy(q_grid, p_grid, h, omega)
exact_energy_grid = harmonic_energy(q_grid, p_grid, omega)

fig_phase, ax = plt.subplots(figsize=single_panel_square)
ax.contour(
    q_grid, p_grid, exact_energy_grid,
    levels=np.linspace(0.2, 2.0, 10), colors="0.82", linewidths=0.6,
)
ax.contour(
    q_grid, p_grid, exact_energy_grid, levels=[modified_energy_0],
    colors="0.4", linewidths=1.5,
)
ax.contour(
    q_grid, p_grid, modified_energy_grid, levels=[modified_energy_0],
    colors="C0", linewidths=1.5,
)
snapshot_indices = np.linspace(0, 10, dtype=int)
points = ax.scatter(
    q[snapshot_indices], p[snapshot_indices],
    c=t[snapshot_indices], cmap="plasma", s=13, zorder=3,
)
ax.plot([], [], color="C0", linewidth=1.5, label=r"$\widetilde H_h(q,p)=\widetilde H_h(q_0,p_0)$")
ax.set(xlabel="$q$", ylabel="$p$", xlim=(-limit, limit), ylim=(-limit, limit))
# ax.set_title(r"Bahn auf $\widetilde H_h(q,p)=\mathrm{const.}$")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend()
colorbar = fig_phase.colorbar(points, ax=ax, pad=0.03, shrink=0.5, fraction=0.05)
colorbar.set_label("Zeit $t$")
fig_phase.tight_layout()
plt.show()

fig_phase.savefig(output_dir / "modified_energy_ellipse.pdf")

In [ ]:
fig_energy, ax = plt.subplots(figsize=single_panel_wide)
ax.plot(t, energy, label="$H$")
ax.plot(t, modified_energy, label=r"$\widetilde H_h$")
ax.set(xlabel="Zeit $t$", ylabel="Energie")
# ax.set_title("Exakte und modifizierte Energie")
ax.grid(alpha=0.3)
ax.legend()
fig_energy.tight_layout()
plt.show()

fig_energy.savefig(output_dir / "modified_energy_conservation.pdf")

In [ ]:
fig, ax = plt.subplots(figsize=single_panel_square)
ax.contour(
    q_grid, p_grid, exact_energy_grid,
    levels=np.linspace(0.2, 2.0, 10), colors="0.85", linewidths=0.7,
)
ax.contour(
    q_grid, p_grid, modified_energy_grid, levels=[modified_energy_0],
    colors="C0", linewidths=1.5,
)
trajectory, = ax.plot([], [], "-o", color="C1", markersize=2.2, linewidth=0.8)
current_point, = ax.plot([], [], "o", color="C3", markersize=5)
step_text = ax.text(0.03, 0.96, "", transform=ax.transAxes, va="top")
ax.set(xlabel="$q$", ylabel="$p$", xlim=(-limit, limit), ylim=(-limit, limit))
# ax.set_title(r"Symplektischer Euler und $\widetilde H_h$")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)

def update_oscillator(frame):
    trajectory.set_data(q[:frame + 1], p[:frame + 1])
    current_point.set_data([q[frame]], [p[frame]])
    step_text.set_text(f"n = {frame},   t = {frame * h:.1f}")
    return trajectory, current_point, step_text

oscillator_animation = FuncAnimation(
    fig, update_oscillator, frames=n_steps + 1, interval=80, blit=True
)
oscillator_video = output_dir / "symplectic_euler_oscillator.mp4"
oscillator_animation.save(str(oscillator_video), writer="ffmpeg", fps=15)
plt.close(fig)
display(Video(
    str(oscillator_video), embed=True,
    html_attributes=f'controls loop muted width="{animation_display_width}"',
))

## Chirikov-Standardabbildung

Die **Chirikov-Standardabbildung** (*Chirikov standard map*) lautet

$$P_{n+1}=P_n-K\sin(q_n), \qquad q_{n+1}=q_n+P_{n+1}. $$

Sie entsteht aus einer symplektischen Euler-Diskretisierung des Pendels; Für die Diskretisierung ist $K=h^2$.

Da $q$ ein Winkel ist, können wir $q$ nach jedem Schritt modulo $2\pi$ betrachten. Der Impuls $P$ kann dagegen beliebig groß werden. Der natürliche Phasenraum ist also wie beim Pendel ein Zylinder. Da die Abbildung periodisch unter der verschiebung $P tp \P+2\pi$ ist,  kann man auch $P$ modulo $2\pi$ betrachten. Die Abbildung lässt sich dann auf einem Torus darstellen. 

Für kleine $K$ approximiert die numerische Bewegung die exakte Pendelbewegung gut; viele Trajektorien ähneln den Energiekonturen des kontinuierlichen Pendels. Bei größeren $K$ treten neben regulären Trajektorien chaotische Bereiche auf: Benachbarte Anfangsbedingungen können sich völlig unterschiedlich entwickeln, obwohl das kontinuierliche Pendel integrabel ist.

In [ ]:
two_pi = 2 * np.pi

def standard_map(q, momentum, K, n_iterations):
    q_history = np.empty(n_iterations)
    momentum_history = np.empty(n_iterations)

    for n in range(n_iterations):
        momentum = momentum - K * np.sin(q)
        q = q + momentum
        q_history[n] = (q+np.pi) % two_pi-np.pi
        momentum_history[n] = (momentum + np.pi) % two_pi - np.pi

    return q_history, momentum_history

In [ ]:
K_values = [0.5, 2, 5.0]
n_initial_conditions = 20
n_iterations = 10000
rng = np.random.default_rng(2627)
initial_q = rng.uniform(0, two_pi, n_initial_conditions)
initial_momentum = rng.uniform(0, two_pi, n_initial_conditions)
trajectory_colors = plt.colormaps["turbo"](
    np.linspace(0.05, 0.95, n_initial_conditions)
)

fig_standard, axes = plt.subplots(
    1, 3, figsize=full_width_three_panel, sharex=True, sharey=True
)

for ax, K in zip(axes, K_values):
    for q_start, momentum_start, color in zip(
        initial_q, initial_momentum, trajectory_colors
    ):
        q_map, momentum_map = standard_map(
            q_start, momentum_start, K, n_iterations
        )
        ax.plot(
            q_map, momentum_map, ".", color=color, markersize=0.3, alpha=0.8,
            rasterized=True, zorder=1,
        )

    ax.text(
        0.05, 0.94, f"$K={K:g}$", transform=ax.transAxes, va="top", zorder=2,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85},
    )
    ax.set_xlim(-np.pi, np.pi)
    ax.set_ylim(-np.pi, np.pi)
    ax.set_xticks([-np.pi,0, np.pi], ["-$\pi$", "$0$", "$\pi$"])
    ax.set_yticks([-np.pi,0, np.pi], ["-$\pi$", "$0$", "$\pi$"])
    ax.set_xlabel(r"$q\;\mathrm{mod}\;2\pi$")

axes[0].set_ylabel(r"$P\;\mathrm{mod}\;2\pi$")
# fig_standard.suptitle("Chirikov-Standardabbildung")
fig_standard.tight_layout(w_pad=0.7)
plt.show()

fig_standard.savefig(output_dir / "chirikov_standard_map.pdf")

### Animation einer einzelnen Trajektorie

Die folgende Funktion baut eine Trajektorie der Standardabbildung schrittweise auf. Anfangszustand $(q_0,P_0)$, Parameter $K$ und Anzahl der Iterationen werden beim Funktionsaufruf festgelegt. Beide Koordinaten werden modulo $2\pi$ auf dem Torus dargestellt.

In [ ]:
def animate_standard_map(q0, momentum0, K, n_iterations=500, interval=30):
    """Animiert eine Trajektorie der Chirikov-Standardabbildung."""
    q_map, momentum_map = standard_map(q0, momentum0, K, n_iterations)
    q_path = np.concatenate(([(q0 + np.pi) % two_pi - np.pi], q_map))
    momentum_path = np.concatenate((
        [(momentum0 + np.pi) % two_pi - np.pi], momentum_map
    ))

    fig, ax = plt.subplots(figsize=single_panel_square)
    trail, = ax.plot([], [], ".", color="C0", markersize=2.2, alpha=0.75)
    current, = ax.plot([], [], "o", color="C3", markersize=5)
    step_text = ax.text(0.03, 0.97, "", transform=ax.transAxes, va="top")
    ax.set(
        xlim=(-np.pi, np.pi), ylim=(-np.pi, np.pi),
        xlabel=r"$q\;\mathrm{mod}\;2\pi$",
        ylabel=r"$P\;\mathrm{mod}\;2\pi$",
        title=rf"Chirikov-Abbildung, $K={K:g}$",
    )
    ax.set_xticks([-np.pi, 0, np.pi], [r"$-\pi$", "$0$", r"$\pi$"])
    ax.set_yticks([-np.pi, 0, np.pi], [r"$-\pi$", "$0$", r"$\pi$"])
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)

    def update(frame):
        trail.set_data(q_path[:frame + 1], momentum_path[:frame + 1])
        current.set_data([q_path[frame]], [momentum_path[frame]])
        step_text.set_text(f"n = {frame}")
        return trail, current, step_text

    animation = FuncAnimation(
        fig, update, frames=n_iterations + 1, interval=interval, blit=True
    )
    video_path = output_dir / "chirikov_trajectory.mp4"
    fps = max(1, round(1000 / interval))
    animation.save(str(video_path), writer="ffmpeg", fps=fps)
    plt.close(fig)
    return Video(
        str(video_path), embed=True,
        html_attributes=f'controls loop muted width="{animation_display_width}"',
    )

In [ ]:
# Parameter der Animation
q0_animation = 0.2
momentum0_animation = 0.0
K_animation = 2.0
n_iterations_animation = 500

display(animate_standard_map(
    q0_animation, momentum0_animation, K_animation,
    n_iterations=n_iterations_animation,
))

### Deformation einer kleinen Fläche im chaotischen Gebiet

Um die Flächenerhaltung und das chaotische Strecken und Falten sichtbar zu machen, verwenden wir eine kleine katzenförmige Punktwolke nahe dem hyperbolischen Fixpunkt. Die Farbe markiert die ursprüngliche vertikale Position eines Punktes und wird bei der Iteration mitgeführt.

In [ ]:
def _points_in_triangle(x, y, vertices):
    """Boolesche Maske für Punkte innerhalb eines Dreiecks."""
    (x1, y1), (x2, y2), (x3, y3) = vertices
    denominator = (y2 - y3)*(x1 - x3) + (x3 - x2)*(y1 - y3)
    a = ((y2 - y3)*(x - x3) + (x3 - x2)*(y - y3)) / denominator
    b = ((y3 - y1)*(x - x3) + (x1 - x3)*(y - y3)) / denominator
    c = 1.0 - a - b
    return (a >= 0) & (b >= 0) & (c >= 0)


def cat_point_cloud(n_points=12000, scale=0.32, center=(0.0, 0.0), seed=2627):
    """Erzeugt eine kleine, gefüllte Katzen-Silhouette als Punktwolke."""
    rng = np.random.default_rng(seed)
    accepted_x, accepted_y = [], []

    while sum(len(values) for values in accepted_x) < n_points:
        x = rng.uniform(-1.15, 1.15, 4*n_points)
        y = rng.uniform(-1.25, 1.05, 4*n_points)

        head = (x/0.58)**2 + ((y - 0.38)/0.48)**2 <= 1.0
        body = (x/0.48)**2 + ((y + 0.48)/0.72)**2 <= 1.0
        left_ear = _points_in_triangle(
            x, y, [(-0.52, 0.55), (-0.38, 1.02), (-0.08, 0.73)]
        )
        right_ear = _points_in_triangle(
            x, y, [(0.08, 0.73), (0.38, 1.02), (0.52, 0.55)]
        )
        feet = (((x + 0.28)/0.24)**2 + ((y + 1.08)/0.18)**2 <= 1.0) | (
            ((x - 0.28)/0.24)**2 + ((y + 1.08)/0.18)**2 <= 1.0
        )
        radius = np.sqrt((x - 0.46)**2 + (y + 0.45)**2)
        angle = np.arctan2(y + 0.45, x - 0.46)
        tail = (radius > 0.52) & (radius < 0.68) & (angle > -0.45) & (angle < 1.55)

        silhouette = head | body | left_ear | right_ear | feet | tail
        left_eye = ((x + 0.21)/0.09)**2 + ((y - 0.46)/0.12)**2 < 1.0
        right_eye = ((x - 0.21)/0.09)**2 + ((y - 0.46)/0.12)**2 < 1.0
        nose = (np.abs(x) < 0.055) & (y > 0.25) & (y < 0.34)
        mask = silhouette & ~(left_eye | right_eye | nose)
        accepted_x.append(x[mask])
        accepted_y.append(y[mask])

    x = np.concatenate(accepted_x)[:n_points]
    y = np.concatenate(accepted_y)[:n_points]
    q = center[0] + scale*x
    momentum = center[1] + scale*y
    color = (y - y.min()) / (y.max() - y.min())
    return q, momentum, color


def standard_map_snapshots(q0, momentum0, K, snapshot_steps):
    """Propagiert eine Punktwolke und speichert ausgewählte Iterationen."""
    steps = sorted(set(snapshot_steps))
    q = np.asarray(q0, dtype=float).copy()
    momentum = np.asarray(momentum0, dtype=float).copy()
    snapshots = {}
    if 0 in steps:
        snapshots[0] = (q.copy(), momentum.copy())

    for n in range(1, max(steps) + 1):
        momentum = momentum - K*np.sin(q)
        q = q + momentum
        q = (q + np.pi) % two_pi - np.pi
        momentum = (momentum + np.pi) % two_pi - np.pi
        if n in steps:
            snapshots[n] = (q.copy(), momentum.copy())
    return snapshots

In [ ]:
# Parameter der Ensemble-Animation
K_cat = 5.0
cat_center = (0.0, 0.0)
cat_scale = 0.32
cat_snapshot_steps = [0, 1, 2, 3, 5, 8]
cat_animation_steps = [0, 1, 2, 3, 4, 5, 6, 8, 10, 12]

q_cat, momentum_cat, cat_color = cat_point_cloud(
    scale=cat_scale, center=cat_center
)
all_cat_steps = sorted(set(cat_snapshot_steps + cat_animation_steps))
cat_snapshots = standard_map_snapshots(
    q_cat, momentum_cat, K_cat, all_cat_steps
)

# Statische Sechs-Panel-Abbildung für das Skript
fig_cat, axes = plt.subplots(
    2, 3, figsize=(latex_textwidth_in, 0.68*latex_textwidth_in),
    sharex=True, sharey=True,
)
for ax, step in zip(axes.flat, cat_snapshot_steps):
    q_snapshot, momentum_snapshot = cat_snapshots[step]
    ax.scatter(
        q_snapshot, momentum_snapshot, c=cat_color, cmap="viridis",
        s=0.35, linewidths=0, rasterized=True,
    )
    ax.text(0.04, 0.94, f"$n={step}$", transform=ax.transAxes, va="top")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-np.pi, np.pi)
    ax.set_ylim(-np.pi, np.pi)
    ax.set_xticks([-np.pi, 0, np.pi], [r"$-\pi$", "$0$", r"$\pi$"])
    ax.set_yticks([-np.pi, 0, np.pi], [r"$-\pi$", "$0$", r"$\pi$"])

for ax in axes[-1, :]:
    ax.set_xlabel(r"$q\;\mathrm{mod}\;2\pi$")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$P\;\mathrm{mod}\;2\pi$")
fig_cat.suptitle(rf"Deformation einer kleinen Fläche, $K={K_cat:g}$")
fig_cat.tight_layout(w_pad=0.4, h_pad=0.5)
plt.show()
fig_cat.savefig(output_dir / "chirikov_cat_evolution.pdf")

In [ ]:
# Langsame Animation derselben Punktwolke
fig, ax = plt.subplots(figsize=single_panel_square)
cat_scatter = ax.scatter(
    [], [], c=[], cmap="viridis", s=0.55, linewidths=0,
    vmin=0.0, vmax=1.0, rasterized=True,
)
step_text = ax.text(0.03, 0.97, "", transform=ax.transAxes, va="top")
ax.set(
    xlim=(-np.pi, np.pi), ylim=(-np.pi, np.pi),
    xlabel=r"$q\;\mathrm{mod}\;2\pi$",
    ylabel=r"$P\;\mathrm{mod}\;2\pi$",
    title=rf"Deformation im chaotischen Gebiet, $K={K_cat:g}$",
)
ax.set_xticks([-np.pi, 0, np.pi], [r"$-\pi$", "$0$", r"$\pi$"])
ax.set_yticks([-np.pi, 0, np.pi], [r"$-\pi$", "$0$", r"$\pi$"])
ax.set_aspect("equal", adjustable="box")

def update_cat(frame):
    step = cat_animation_steps[frame]
    q_snapshot, momentum_snapshot = cat_snapshots[step]
    cat_scatter.set_offsets(np.column_stack((q_snapshot, momentum_snapshot)))
    cat_scatter.set_array(cat_color)
    step_text.set_text(f"n = {step}")
    return cat_scatter, step_text

cat_animation = FuncAnimation(
    fig, update_cat, frames=len(cat_animation_steps), interval=1200, blit=True
)
cat_video = output_dir / "chirikov_cat_evolution.mp4"
cat_animation.save(str(cat_video), writer="ffmpeg", fps=1)
plt.close(fig)
display(Video(
    str(cat_video), embed=True,
    html_attributes=f'controls loop muted width="{animation_display_width}"',
))